# browniano 1 — prueba de la grabación de ThorCam

Grabación `C:\datos_pinzas\GRABACIONES\browniano 1` (50 % glicerina – 50 % agua, **pinza apagada**). Es para probar dos cosas:

1. que el **formato** de ThorCam sirve (cuadros, tamaño, tiempo);
2. que el procesamiento de TouchDesigner (B0 → B8, corrido en lote con `_lote.py`) **reconoce las beads y las sigue** todo el video.

Se espera que las beads estén pegadas al vidrio: el MSD tiene que salir plano, al nivel del ruido de localización.

Parámetros usados en los bloques (distintos de los defaults para MP4): B0 `thorcam` con 13.669 fps; B2 **espacial** (con mediana las beads quietas se borran y no se detecta ninguna); B6 umbral **4** (con 12 se detectaba una sola bead).

In [ ]:
from trayectorias import *
import glob
import re
from pinzas import fisica, trampa

usar_estilo()

ORIGINAL = r"C:\datos_pinzas\GRABACIONES\browniano 1"
CARPETA = max(glob.glob(r"C:\datos_pinzas\IMPORTADAS\*_importada_50G-50W\csv\*"), key=os.path.getmtime)
LARGO_MIN = 1000        # cuadros medidos: las trayectorias más cortas no entran
MAX_LAG = 300           # retardo máximo del MSD, en cuadros (~22 s)

datos = cargar(CARPETA)
t, X, Y, ids, u = matrices(datos, largo_min=LARGO_MIN)
print(CARPETA)
print('%d cuadros, %.1f s | %d partículas con más de %d cuadros (de %d en total) | unidad: %s'
      % (len(t), t[-1] - t[0], len(ids), LARGO_MIN, datos['tr'].particula.nunique(), u))

## 1. Formato

ThorCam parte la grabación en archivos de 167 cuadros: `50G-50W.tif`, `50G-50W_1.tif`, … El primero no lleva número. El importador ahora los junta (`pinzas/importar.py`, versión 3); antes tomaba solo el primero.

El TIFF **no trae el tiempo de cada cuadro** (ni contador ni timestamp), así que el tiempo es nominal. Para estimar el fps se usa la hora en que se terminó de escribir cada archivo: se ajusta una recta a esa hora en función de los cuadros acumulados.

In [ ]:
f = datos['meta']['fuente']
cam = datos['meta']['camara']
print('archivos: %d (%s … %s) | cuadros: %d | %d × %d px, %s %s, %d bits'
      % (len(f['archivos']), f['archivos'][0], f['archivos'][-1], sum(f['paginas']),
         f['forma'][1], f['forma'][0], f['fotometrica'], f['dtype'], cam['bits']))
print('tiempo:', datos['meta']['tiempo_fuente'], '| período usado: %.3f ms (%.3f fps)' % (cam['periodo_us'] / 1e3, 1e6 / cam['periodo_us']))

# fps por la hora de escritura de cada archivo (el último, incompleto, no entra)
n_cuadros = np.cumsum(f['paginas'])
hora = np.array([os.path.getmtime(os.path.join(ORIGINAL, a)) for a in f['archivos']])
p, cov = np.polyfit(n_cuadros[:-1], hora[:-1], 1, cov=True)
fps, err_fps = 1 / p[0], np.sqrt(cov[0, 0]) / p[0] ** 2
fps_archivo = np.array(f['paginas'][1:-1]) / np.diff(hora[:-1])
print('fps por la hora de los archivos: %.3f ± %.3f | por archivo: de %.2f a %.2f'
      % (fps, err_fps, fps_archivo.min(), fps_archivo.max()))

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(np.arange(1, len(fps_archivo) + 1), fps_archivo, 'o-', color=AZUL)
ax.axhline(fps, color='k', ls=':', label='ajuste: %.3f fps' % fps)
ax.set_xlabel('archivo'); ax.set_ylabel('fps'); ax.legend()
ax.set_title('cuadros por segundo de cada archivo (167 cuadros / tiempo entre archivos)');

In [ ]:
# ¿cuadros repetidos? (la cámara repite el anterior si no entregó uno nuevo): todas las beads con Δx = Δy = 0 exacto
dX, dY = np.diff(X, axis=0), np.diff(Y, axis=0)
quieto = np.all((dX == 0) & (dY == 0) | ~np.isfinite(dX), axis=1) & np.isfinite(dX).any(1)
print('pasos con todas las beads exactamente quietas (cuadro repetido): %d de %d' % (quieto.sum(), len(dX)))

- Todas las partes tienen 167 cuadros, salvo la última (6). Son **5183 cuadros**, RGB de 8 bits ya procesados por ThorCam: no es el crudo de 10 bits. Para mejores grabaciones conviene *Unprocessed* (ver `docs/configurar_pc.md`).
- El fps de cada archivo es ~13.7. La parte `_17` tardó ~0.5 s más: puede ser una demora al escribir o cuadros perdidos, y sin tiempos por cuadro no se puede saber.

## 2. Detección

Cuántas beads reconoce B6 en cada cuadro y con qué contraste (z = realce / ruido, de B5).

In [ ]:
tr = datos['tr']
por_cuadro = tr[tr.x_px.notna()].groupby('cuadro').size().reindex(range(len(t)), fill_value=0)
z_med = tr.groupby('particula').z.median().loc[ids]
largos = datos['particulas'].set_index('particula')
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(t, por_cuadro.values, color=AZUL, lw=0.8)
axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('beads detectadas')
axs[0].set_title('beads por cuadro: %.1f ± %.1f (mín %d, máx %d)'
                 % (por_cuadro.mean(), por_cuadro.std(), por_cuadro.min(), por_cuadro.max()))
medidos = np.isfinite(X).sum(0)
axs[1].bar(range(len(ids)), 100 * medidos / len(t), color=AZUL)
axs[1].set_xticks(range(len(ids)), ids)
axs[1].set_xlabel('partícula'); axs[1].set_ylabel('% de los cuadros')
axs[1].set_title('en qué fracción del video se sigue cada una')
plt.tight_layout()
pd.DataFrame({'particula': ids, 'cuadros': medidos, '% del video': (100 * medidos / len(t)).round(1),
              'z mediano': z_med.values.round(1), 'x medio [px]': np.nanmean(X, 0).round(0),
              'y medio [px]': np.nanmean(Y, 0).round(0)})

## 3. Trayectorias y deriva

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
graficar_trayectorias(datos, X, Y, ids, ax=axs[0])
graficar_xy_t(t, X, Y, ids, u, axs=None)
ax, v = graficar_deriva(t, X, Y, u, ax=axs[1])
dx, dy, _ = deriva(X, Y)
(px_, cx), (py_, cy) = [np.polyfit(t, d, 1, cov=True) for d in (dx, dy)]
print('deriva común: vx = %.4f ± %.4f, vy = %.4f ± %.4f px/s → %.1f px en todo el video'
      % (px_[0], np.sqrt(cx[0, 0]), py_[0], np.sqrt(cy[0, 0]), np.hypot(dx[-1], dy[-1])))

Las trayectorias son puntos: las beads no se mueven más que unos pocos px en 6 minutos, y lo que se mueven lo hacen **todas juntas**. Es deriva de la muestra, no browniano.

## 4. MSD

Con la deriva común restada. Para comparar, el MSD que tendría una bead **libre** de 3 µm en la mezcla 50/50 (Stokes–Einstein con η de Cheng 2008). Para pasarlo a px se supone una escala por el tamaño aparente de las beads.

Supuestos: diámetro de 3 µm; 50 % **en masa** a 23 °C; diámetro aparente de 37 ± 3 px (perfil radial de 9 beads en los primeros cuadros). Falta la retícula para tener la escala real.

In [ ]:
DIAMETRO_UM, DIAMETRO_PX, ERR_DIAMETRO_PX = 3.0, 37.0, 3.0      # SUPUESTOS
T, FRACCION_GLICERINA = 296.15, 0.50
ESCALA = DIAMETRO_UM / DIAMETRO_PX                               # µm/px
ERR_ESCALA = ESCALA * ERR_DIAMETRO_PX / DIAMETRO_PX
eta = fisica.viscosidad_glicerina_agua(FRACCION_GLICERINA, T)
D_libre = D_teorico(DIAMETRO_UM, T, eta)                         # µm²/s
D_libre_px = D_libre / ESCALA ** 2
print('η = %.2f mPa·s | D libre = %.4f µm²/s = %.1f px²/s | escala supuesta %.0f ± %.0f nm/px'
      % (eta * 1e3, D_libre, D_libre_px, ESCALA * 1e3, ERR_ESCALA * 1e3))

Xc, Yc = sin_deriva(X, Y)
m = msd(t, Xc, Yc, MAX_LAG)
tabla = coeficientes_D(m, n_puntos=4)
alfa = pendiente_loglog(m, desde=10)
ax = graficar_msd(m, u, etiqueta='(deriva común restada)')
tau = m['tau'][np.isfinite(m['tau'])]
ax.plot(tau, 4 * D_libre_px * tau, '--', color=ROJO, label='bead libre: 4Dτ, D = %.1f px²/s' % D_libre_px)
ax.set_title('MSD de %d beads: pendiente log-log %.2f ± %.2f' % (len(ids), np.nanmean(alfa), np.nanstd(alfa)))
ax.legend(fontsize=9);

In [ ]:
mx, my = m['msd_x'], m['msd_y']
print('MSD a τ = %.1f s: %.3g px² (mediana entre beads); una libre tendría %.0f px²'
      % (m['tau'][-1], np.nanmedian(m['msd'][-1]), 4 * D_libre_px * m['tau'][-1]))
print('D aparente (primeros 4 retardos): mediana %.3g px²/s = %.1e de D libre'
      % (np.nanmedian(tabla.D), np.nanmedian(tabla.D) / D_libre_px))

## 5. Ruido de localización

Con las beads pegadas, lo que se mueve la posición de un cuadro al siguiente es el ruido del tracking: $\sigma^2 = \langle (x_{i+1}-x_i)^2\rangle / 2$ (`pinzas.trampa.piso_de_ruido`). Es el piso con el que se va a comparar la $\langle x^2\rangle$ de una bead atrapada.

Error: desvío entre 10 bloques del video / √10.

In [ ]:
N_BLOQUES = 10
filas = []
for k, p in enumerate(ids):
    r = trampa.piso_de_ruido(X[:, k], Y[:, k])
    bloques = [trampa.piso_de_ruido(xb, yb) for xb, yb in zip(np.array_split(X[:, k], N_BLOQUES), np.array_split(Y[:, k], N_BLOQUES))]
    fila = {'particula': p, 'z mediano': z_med.loc[p]}
    for e in 'xy':
        s = np.array([np.sqrt(b['var_' + e]) for b in bloques])
        fila['σ%s [px]' % e] = r['sigma_' + e]
        fila['err σ%s' % e] = np.nanstd(s, ddof=1) / np.sqrt(np.isfinite(s).sum())
    filas.append(fila)
ruido = pd.DataFrame(filas)
s_med = np.sqrt((ruido['σx [px]'] ** 2 + ruido['σy [px]'] ** 2) / 2)
print('σ por eje (mediana entre beads): %.3f px ≈ %.0f ± %.0f nm con la escala supuesta'
      % (np.median(s_med), np.median(s_med) * ESCALA * 1e3, np.median(s_med) * ERR_ESCALA * 1e3))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(ruido['z mediano'], ruido['σx [px]'], ruido['err σx'], fmt='o', color=AZUL, label='x', capsize=2)
ax.errorbar(ruido['z mediano'], ruido['σy [px]'], ruido['err σy'], fmt='s', color=NARANJA, label='y', capsize=2)
ax.set_xlabel('contraste z (mediano)'); ax.set_ylabel('σ de localización [px]'); ax.legend()
ax.set_title('ruido de localización: baja con el contraste')
ruido.round(4)

## Conclusión